# PQI Pipeline , Pressure Quality Index

Computes the **Pressure Quality Index (PQI)** for every player × phase across all 5 Bundesliga matches.

PQI measures how well a player executes pressing actions, combining three sub-scores:
- **Orientation** , body orientation relative to the ball carrier
- **Stance** , body posture readiness during the press
- **Proximity** , distance to the ball carrier at press time

**Output:** `results/pqi_full.csv` , 400 rows (20 players × 2 halves × 5 matches × 2 teams).

**Checkpoint:** completed phases are saved incrementally; re-running skips already-processed phases.

In [ ]:
import os, sys
from pathlib import Path

# Find project root (pyproject.toml marker) and chdir to notebooks/
# so ../results/ and ../figures/ resolve correctly from any launch CWD.
_root = next(
    (p for p in [Path().resolve(), *Path().resolve().parents] if (p / "pyproject.toml").exists()),
    None,
)
if _root is None:
    raise RuntimeError("Cannot locate project root  -  pyproject.toml not found.")
os.chdir(_root / "notebooks")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


## Step 1. Setup: AWS Session and Configuration

Initialises the AWS session and S3 client needed to stream skeleton Parquet files.
Sets the S3 bucket, challenge prefix, and checkpoint path.

In [1]:
import os
import sys
sys.path.insert(0, "..")

from src.eda_helpers import create_session
from src.pressure_pipeline import run_all_matches_pqi, MATCH_CONFIGS

session, s3_client, s3fs = create_session()
BUCKET = os.environ.get("HACKATHON_BUCKET", "your-s3-bucket-name")
PREFIX = "Challenge 2 – Unlock the Power of 3D Football Data/Match_Data"
CHECKPOINT = "../results/pqi_checkpoint.csv"

## Step 2. Run PQI Pipeline

Streams each match's skeleton Parquet from S3 and computes PQI sub-scores per player per phase.

Progress is checkpointed after each phase so the pipeline can be safely interrupted and resumed.

In [2]:
pqi_df = run_all_matches_pqi(
    s3_client,
    s3fs,
    BUCKET,
    PREFIX,
    match_configs=MATCH_CONFIGS,
    checkpoint_path=CHECKPOINT,
)


=== FCB-HSV: FC Bayern Muenchen vs Hamburger SV ===
  [FCB-HSV] 40 players, 2 phases
  [FCB-HSV] Phase 1st half: frames 3,330,943 – 3,484,329 ... 121s
  [FCB-HSV] Phase 1st half done — 40 rows saved to checkpoint.
  [FCB-HSV] Phase 2nd half: frames 3,536,417 – 3,678,119 ... 113s
  [FCB-HSV] Phase 2nd half done — 40 rows saved to checkpoint.
  Done: 80 player-phase rows
  [checkpoint] Saved 80 rows to ../results/pqi_checkpoint.csv

=== BVB-VFB: Borussia Dortmund vs VfB Stuttgart ===
  [BVB-VFB] 40 players, 2 phases
  [BVB-VFB] Phase 1st half: frames 2,790,134 – 2,935,531 ... 130s
  [BVB-VFB] Phase 1st half done — 40 rows saved to checkpoint.
  [BVB-VFB] Phase 2nd half: frames 2,984,098 – 3,141,386 ... 135s
  [BVB-VFB] Phase 2nd half done — 40 rows saved to checkpoint.
  Done: 80 player-phase rows
  [checkpoint] Saved 160 rows to ../results/pqi_checkpoint.csv

=== SGE-FCB: Eintracht Frankfurt vs FC Bayern Muenchen ===
  [SGE-FCB] 40 players, 2 phases
  [SGE-FCB] Phase 1st half: frames 3

## Step 3. Save Results

Writes the full PQI DataFrame to `results/pqi_full.csv` and prints a shape/preview summary.

In [3]:
pqi_df.to_csv("../results/pqi_full.csv", index=False)
print(f"Shape: {pqi_df.shape}")
print(pqi_df.head())

Shape: (400, 17)
   jersey  team                          name position match_id phase_label  \
0      23     1                    Sacha Boey       RV  FCB-HSV    1st half   
1      45     1           Aleksandar Pavlović      DML  FCB-HSV    1st half   
2      27     1                 Konrad Laimer       RV  FCB-HSV    1st half   
3      14     1  Luis Fernando Díaz Marulanda      OLM  FCB-HSV    1st half   
4       3     1                    Minjae Kim           FCB-HSV    1st half   

   phase_start  phase_end   mean_pqi  median_pqi    std_pqi  n_press_frames  \
0      3330943    3484329        NaN         NaN        NaN               0   
1      3330943    3484329  62.731984   63.949942  16.180320          150179   
2      3330943    3484329  59.366913   59.893542  16.750371          152539   
3      3330943    3484329  58.954233   59.605359  16.745884          150663   
4      3330943    3484329        NaN         NaN        NaN               0   

   orientation_mean  stance_mean 

## Step 4. Validation

Asserts that the output has the expected 400 rows and all required columns are present.
Fails loudly if the pipeline produced incomplete or malformed output.

In [4]:
assert len(pqi_df) == 400, f"Expected 400 rows, got {len(pqi_df)}"
required_cols = [
    "jersey", "team", "name", "position", "match_id", "phase_label",
    "phase_start", "phase_end", "mean_pqi", "median_pqi", "std_pqi",
    "n_press_frames", "press_minutes", "orientation_mean", "stance_mean",
    "proximity_mean", "coverage_pct"
]
missing = [c for c in required_cols if c not in pqi_df.columns]
assert not missing, f"Missing columns: {missing}"
print("\u2713 All validations passed")

✓ All validations passed
